In [1]:
import requests
from pydantic import BaseModel, ConfigDict, Field, field_validator, ValidationError, IPvAnyAddress

/Users/sarathv/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
from typing import Optional
class IPGeo(BaseModel):
    model_config = ConfigDict(extra="ignore")

    ip: IPvAnyAddress
    country: Optional[str] = None
    country_code: Optional[str] = Field(default=None, min_length=2, max_length=2)
    country_code3: Optional[str] = Field(default=None, min_length=2, max_length=3)
    city: Optional[str] = None
    region: Optional[str] = None
    timezone: Optional[str] = None
    organization_name: Optional[str] = None

    @field_validator("organization_name", mode="after")
    @classmethod
    def set_unknown_to_none(cls, value: str):
        if value.casefold() == "unknown":
            return None
        return value

In [4]:
IPGeo(ip="8.8.8.8", country="test", country_code3 = "USA", organization_name="Unknown")

IPGeo(ip=IPv4Address('8.8.8.8'), country='test', country_code=None, country_code3='USA', city=None, region=None, timezone=None, organization_name=None)

In [5]:
url_query = "https://get.geojs.io/v1/geo/{ip_address}.json"

In [6]:
url = url_query.format(ip_address="8.8.8.8")
response = requests.get(url)
response.raise_for_status()

In [7]:
response_json= response.json()

In [8]:
print(response.json())

{'accuracy': 1000, 'area_code': '0', 'asn': 15169, 'continent_code': 'NA', 'country': 'United States', 'country_code': 'US', 'country_code3': 'USA', 'ip': '8.8.8.8', 'latitude': '37.751', 'longitude': '-97.822', 'organization': 'AS15169 Google LLC', 'organization_name': 'Google LLC', 'timezone': 'America/Chicago'}


In [9]:
data = IPGeo.model_validate(response.json())

In [10]:
url = url_query.format(ip_address="23.62.177.155")
response = requests.get(url)
response.raise_for_status()
data = IPGeo.model_validate(response.json())
print(data.model_dump_json(indent=2))

{
  "ip": "23.62.177.155",
  "country": "United States",
  "country_code": "US",
  "country_code3": "USA",
  "city": "El Segundo",
  "region": "California",
  "timezone": "America/Los_Angeles",
  "organization_name": "Akamai Technologies, Inc."
}
